# Serve the shop assistant model on a free Colab T4

Runtime → Change runtime type → **T4 GPU**, then fill in the two settings below and run all cells top to bottom.

Settings used: context **1,024** tokens (longest prompt 216 + longest reply 177 = 393, leaving ~630 for stock rows; the biggest product has 14 variants ≈ 560 tokens), **fp16** (T4 has no bf16), **20** concurrent sequences.

## Settings
The exact agreed model and your repo. Nothing else needs editing.

In [ ]:
MODEL_NAME = ""   # exact Hugging Face ID of the agreed model
REPO = "ma7721676-ops/maryam"

assert MODEL_NAME, "Set MODEL_NAME to the agreed model"
assert REPO, "Set REPO to your GitHub owner/name"
import os
os.environ["MODEL_NAME"] = MODEL_NAME

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi
!git clone https://github.com/{REPO}.git repo
%cd repo
# Data CSVs live in Google Drive (MyDrive/shop-assistant-data/); copy them in if the repo has none
import os, glob
if not os.path.exists("data/shopper_question_replay.csv"):
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("data", exist_ok=True)
    !cp /content/drive/MyDrive/shop-assistant-data/*.csv data/
print(sorted(glob.glob("data/*.csv")))
!pip install -q -r requirements.txt

## 2. Start the vLLM server in the background
Port 8001, OpenAI-compatible. Waits until `/v1/models` answers (first start downloads the model, so this can take several minutes).

In [ ]:
import subprocess, time, httpx

vllm = subprocess.Popen(
    ["vllm", "serve", MODEL_NAME,
     "--port", "8001",
     "--dtype", "half",
     "--max-model-len", "1024",
     "--max-num-seqs", "20",
     "--gpu-memory-utilization", "0.90"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(180):
    try:
        if httpx.get("http://localhost:8001/v1/models", timeout=2).status_code == 200:
            print("vLLM is up"); break
    except Exception:
        pass
    if vllm.poll() is not None:
        raise RuntimeError("vLLM exited, see vllm.log:\n" + open("vllm.log").read()[-3000:])
    time.sleep(5)
else:
    raise RuntimeError("vLLM did not start in 15 minutes, see vllm.log")

## 3. Send one test question straight to vLLM

In [ ]:
from openai import OpenAI
llm = OpenAI(base_url="http://localhost:8001/v1", api_key="not-needed")
r = llm.chat.completions.create(
    model=MODEL_NAME, max_tokens=60,
    messages=[{"role": "user", "content": "Is the Ember Merino Beanie warm enough for winter?"}],
)
print(r.choices[0].message.content)
print(r.usage)

## 4. Start the FastAPI endpoint
Port 8000, one worker (one GPU, one model server). Concurrency comes from async calls plus a semaphore of 20.

In [ ]:
api = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],
    stdout=open("api.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        if httpx.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("API is up:", httpx.get("http://localhost:8000/health").json()); break
    except Exception:
        pass
    time.sleep(2)
else:
    raise RuntimeError("API did not start, see api.log:\n" + open("api.log").read()[-3000:])

## 5. Load test: 20 shoppers at once
Picks 20 questions with seed 42, sends them together, and saves `results/load_test_<date>.jsonl` plus a summary with p50/p95 latency, tokens/sec and peak GPU memory.

In [ ]:
!python tests/load_test.py --n 20 --seed 42

## 6. Check the results file
20 lines, every `question_id` from the replay CSV, the right catalog rows for each `product_sku` + `variant_size`, all `ok`, and all 20 overlapping in time.

In [ ]:
!python tests/check_results.py

## 7. Show the saved results
Prints the results file and summary so they can be added to the repo.

In [ ]:
import glob
jsonl = sorted(glob.glob("results/load_test_*[0-9].jsonl"))[-1]
print("FILE:", jsonl); print(open(jsonl).read())
summ = jsonl.replace(".jsonl", "_summary.json")
print("FILE:", summ); print(open(summ).read())